# Add an `audio_mp3` preview column to a Lance dataset

End-to-end smoke run for `synth_setter.pipeline.data.add_mp3_audio`:

1. Build a tiny Lance dataset with the same schema `generate-dataset` / `finalize-dataset` write (`audio` / `mel_spec` / `param_array` tensor columns plus embedded `ShardMetadata`).
2. Run the MP3 add step, which appends an `audio_mp3` Lance blob v2 column (tagged `mime_type: audio/mpeg`).
3. Read back a dataframe of the params alongside each row's MP3 size, then decode one blob.

In [ ]:
import io
import tempfile
from pathlib import Path

import lance
import numpy as np
from pedalboard.io import AudioFile

from synth_setter.data.vst.shapes import AUDIO_FIELD, MEL_SPEC_FIELD, PARAM_ARRAY_FIELD
from synth_setter.pipeline.data.add_mp3_audio import (
    AUDIO_MP3_FIELD,
    add_mp3_audio_column,
)
from synth_setter.pipeline.data.lance_shard import (
    lance_schema,
    record_batch_from_arrays,
    write_lance_dataset,
)
from synth_setter.pipeline.schemas.shard_metadata import ShardMetadata

## 1. Build a smoke Lance dataset

Four rows of stereo sine tones (one pitch per row) plus dummy mel/param tensors. The sample rate lives in the shard metadata, exactly as the real writers emit it.

In [ ]:
SAMPLE_RATE = 16000
CHANNELS = 2
DURATION_SECONDS = 0.25
TIME_SAMPLES = int(SAMPLE_RATE * DURATION_SECONDS)
NUM_PARAMS = 5
FREQS = [220.0, 330.0, 440.0, 660.0]
ROWS = len(FREQS)

field_shapes = {
    AUDIO_FIELD: (ROWS, CHANNELS, TIME_SAMPLES),
    MEL_SPEC_FIELD: (ROWS, CHANNELS, 4, 4),
    PARAM_ARRAY_FIELD: (ROWS, NUM_PARAMS),
}

t = np.arange(TIME_SAMPLES) / SAMPLE_RATE
mono = np.stack([0.5 * np.sin(2 * np.pi * f * t) for f in FREQS])[:, None, :]
audio = np.broadcast_to(mono, field_shapes[AUDIO_FIELD]).astype(np.float16)

arrays = {
    AUDIO_FIELD: audio,
    MEL_SPEC_FIELD: np.zeros(field_shapes[MEL_SPEC_FIELD], dtype=np.float32),
    # Distinct per-row params so the final dataframe is legible.
    PARAM_ARRAY_FIELD: np.linspace(0, 1, ROWS * NUM_PARAMS, dtype=np.float32).reshape(
        field_shapes[PARAM_ARRAY_FIELD]
    ),
}

metadata = ShardMetadata(
    velocity=100,
    signal_duration_seconds=DURATION_SECONDS,
    sample_rate=SAMPLE_RATE,
    channels=CHANNELS,
    min_loudness=-55.0,
)

tmpdir = tempfile.mkdtemp(prefix="mp3-audio-smoke-")
uri = Path(tmpdir) / "shard-000000.lance"
schema = lance_schema(field_shapes, metadata)
write_lance_dataset(uri, schema, [record_batch_from_arrays(arrays, schema)])

print("wrote", uri)
lance.dataset(str(uri)).schema

## 2. Run the MP3 add step

`add_mp3_audio_column` reads the sample rate from the shard metadata and backfills an `audio_mp3` Lance blob v2 column in place (committing a new dataset version), tagged `mime_type: audio/mpeg`.

In [ ]:
add_mp3_audio_column(uri, bitrate_kbps=128)

lance.dataset(str(uri)).schema

## 3. Dataframe of just the params and the MP3 bytes

In [ ]:
ds = lance.dataset(str(uri))
# Blob v2 columns are read lazily via take_blobs, not to_table (which yields
# blob descriptors rather than bytes).
mp3_payloads = [
    blob.readall()
    for blob in ds.take_blobs(blob_column=AUDIO_MP3_FIELD, indices=list(range(ds.count_rows())))
]
df = ds.to_table(columns=[PARAM_ARRAY_FIELD]).to_pandas()
df["mp3_bytes"] = [len(p) for p in mp3_payloads]
df

### Sanity check: decode one MP3 cell back to audio

Confirms the bytes are a real MP3 stream at the metadata sample rate and channel count.

In [ ]:
payload = mp3_payloads[0]
with AudioFile(io.BytesIO(payload)) as f:
    decoded = f.read(f.frames)
    print(
        "sample_rate:",
        int(f.samplerate),
        "channels:",
        decoded.shape[0],
        "frames:",
        decoded.shape[1],
    )